# 數學解惑：第二階段訓練（有溫度、會用工具的家教 AI）

**從上往下，一格一格按執行就好。** 全部大約 1.5～2 小時。

- **Kaggle**：右上角 Save Version → Save & Run All，可以關掉 App，跑完再回來看。
- **Colab**：要讓頁面一直開著（手機接上充電、自動鎖定設成永不）。

開始前：
1. 開啟 GPU。Kaggle 在右側 Session options → Accelerator 選 GPU，並打開 Internet；Colab 在「執行階段 → 變更執行階段類型」選 GPU。
2. 把 Hugging Face 的寫入金鑰放進密鑰，名稱叫 `HF_TOKEN`。Kaggle 在 Add-ons → Secrets；Colab 在左邊的鑰匙圖示。**不要把金鑰貼在程式碼裡。**


## 1. 檢查 GPU

In [ ]:
import torch
print('GPU：', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '沒有 GPU！請先開啟 GPU 再重新執行')

## 2. 安裝套件（約 1～2 分鐘）
如果這格之後出現奇怪的錯誤，請「重新啟動工作階段」，然後從第 3 格開始執行。

In [ ]:
# 用比較舊、比較穩的版本：不含只支援新顯示卡的加速功能，T4 也能跑
# Kaggle 內建的舊版 torchao 會讓 LoRA 出錯，先移除（用不到它）
!pip uninstall -y -q torchao
!pip install -q "transformers==4.56.2" "trl==0.23.1" "peft==0.17.1" datasets accelerate huggingface_hub

## 3. 下載專案的訓練資料與程式

In [ ]:
import os
# 每次都抓最新版（舊的資料夾留著會跑到舊程式）
!rm -rf repo && git clone -q --depth 1 https://github.com/tsaibohau/-AI.git repo
!cd repo && git log --oneline -1
print('資料：', os.listdir('repo/training/chat/data'))

## 4. 讀取 Hugging Face 金鑰（從密鑰，不會顯示出來）

In [ ]:
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
    except Exception:
        pass
if token:
    os.environ['HF_TOKEN'] = token
    print('已讀到金鑰')
else:
    print('沒有讀到金鑰：訓練照樣會跑，只是最後不會上傳到 Hugging Face')

## 5. 訓練（最久的一步）
會依序做：訓練前考試 → 微調 → 偏好訓練 → 訓練後考試 → 上傳。中途斷線的話，重新執行這一格會從上次做完的地方接著跑。

In [ ]:
import subprocess, sys
OUT = '/kaggle/working/out' if os.path.exists('/kaggle/working') else '/content/out'
os.environ['OUT'] = OUT
os.makedirs(OUT, exist_ok=True)
LOG = f'{OUT}/train.log'
# 完整記錄存到 train.log（Output 分頁 → out → train.log）；失敗時把錯誤印在最下面
with open(LOG, 'a', encoding='utf-8') as f:
    proc = subprocess.Popen([sys.executable, 'training/chat/train.py', '--out', OUT], cwd='repo', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        f.write(line)
        f.flush()
        # 畫面只印重點（時間開頭的進度、loss），完整內容在 train.log
        if line[:2].isdigit() and line[2:3] == ':' or "'loss'" in line:
            print(line.strip()[:300])
if proc.wait() != 0:
    tail = open(LOG, encoding='utf-8').read().splitlines()[-40:]
    print('\n' + '=' * 30 + '\n訓練失敗，請把下面這段截圖給 Claude：\n' + '=' * 30)
    print('\n'.join(tail))
    raise SystemExit('訓練失敗（錯誤在上面）')

## 6. 看結果：訓練前後的考試對照

In [ ]:
import json
b = json.load(open(f'{OUT}/exam_before.json'))
a = json.load(open(f'{OUT}/exam_after.json'))
print(f"自動檢查通過：訓練前 {b['ok']}/{b['total']} → 訓練後 {a['ok']}/{a['total']}\n")
for rb, ra in zip(b['rows'], a['rows']):
    print('學生：', rb['user'])
    print('  前：', rb['reply'][:120].replace('\n', ' '), '' if not rb['problems'] else '⚠' + '、'.join(rb['problems']))
    print('  後：', ra['reply'][:120].replace('\n', ' '), '' if not ra['problems'] else '⚠' + '、'.join(ra['problems']))
    print()

## 7. 轉成手機版並公開上傳（約 5～10 分鐘）
網站要能下載，所以手機版是**公開**的（使用者已同意）；原始模型仍是私人。轉法已和 mlc-ai 官方手機版逐位元比對過。

In [ ]:
!cd repo && python training/chat/to_mlc.py --out $OUT

## 8. 架伺服器版（學生不用下載，約 10 分鐘）
轉成 GGUF、公開上傳，並建立免費的 Hugging Face Space 跑 llama.cpp。跑完後 Space 還要自己建置 5～10 分鐘。

In [ ]:
!cd repo && python training/chat/to_server.py --out $OUT